<a href="https://colab.research.google.com/github/harshitha020505/DLLAB/blob/main/DlAssignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import keras
from keras import datasets
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from keras.utils import to_categorical
from keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator


(x_train, y_train), (x_test, y_test) = datasets.cifar100.load_data()

x_train_main = x_train[:40000]
y_train_main = y_train[:40000]
x_val = x_train[40000:]
y_val = y_train[40000:]

x_train_main = x_train_main / 255.0
x_val = x_val / 255.0
x_test = x_test / 255.0

y_train_main = to_categorical(y_train_main, 100)
y_val = to_categorical(y_val, 100)
y_test = to_categorical(y_test, 100)

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

datagen.fit(x_train_main)

model = Sequential()

model.add(Conv2D(32, (5,5), activation='relu', kernel_initializer='he_normal', input_shape=(32,32,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(64, (5,5), activation='relu', kernel_initializer='he_normal'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))

model.add(Flatten())

model.add(Dense(120, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(84, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(100, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.0001),
    metrics=['accuracy']
)

model.fit(
    datagen.flow(x_train_main, y_train_main, batch_size=128),
    epochs=40,
    validation_data=(x_val, y_val)
)

test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy:", test_acc)


Among all the above code i got better if the le-net uses maxPooling and relu activation function



In [1]:
import keras
from keras import datasets
from keras.models import Sequential, Model
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Input, Add, Activation
from keras.utils import to_categorical
from keras.optimizers import Adam

# Load CIFAR-100
(x_train, y_train), (x_test, y_test) = datasets.cifar100.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# One-hot
y_train = to_categorical(y_train, 100)
y_test = to_categorical(y_test, 100)

# Split
x_train_main = x_train[:40000]
y_train_main = y_train[:40000]
x_val = x_train[40000:]
y_val = y_train[40000:]


# =========================
# 🔹 LeNet Model
# =========================
def build_lenet(drop=False):
    model = Sequential()

    model.add(Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)))
    model.add(MaxPooling2D((2,2)))

    model.add(Conv2D(64, (3,3), activation='relu'))
    model.add(MaxPooling2D((2,2)))

    model.add(Flatten())

    model.add(Dense(120, activation='relu'))
    if drop:
        model.add(Dropout(0.5))

    model.add(Dense(84, activation='relu'))
    if drop:
        model.add(Dropout(0.5))

    model.add(Dense(100, activation='softmax'))

    return model


# =========================
# 🔹 ZFNet Model
# =========================
def build_zfnet(drop=False):
    model = Sequential()

    model.add(Conv2D(64, (3,3), activation='relu', padding='same', input_shape=(32,32,3)))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2,2)))

    model.add(Conv2D(192, (3,3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2,2)))

    model.add(Conv2D(384, (3,3), activation='relu', padding='same'))
    model.add(Conv2D(256, (3,3), activation='relu', padding='same'))
    model.add(Conv2D(256, (3,3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2,2)))

    model.add(Flatten())

    model.add(Dense(512, activation='relu'))
    if drop:
        model.add(Dropout(0.5))

    model.add(Dense(256, activation='relu'))
    if drop:
        model.add(Dropout(0.5))

    model.add(Dense(100, activation='softmax'))

    return model


# =========================
# 🔹 ResNet Block
# =========================
def res_block(x, filters):
    shortcut = x

    x = Conv2D(filters, (3,3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2D(filters, (3,3), padding='same')(x)
    x = BatchNormalization()(x)

    x = Add()([x, shortcut])
    x = Activation('relu')(x)

    return x


# =========================
# 🔹 ResNet Model
# =========================
def build_resnet(drop=False):
    inputs = Input(shape=(32,32,3))

    x = Conv2D(64, (3,3), padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = res_block(x, 64)
    x = res_block(x, 64)

    x = MaxPooling2D((2,2))(x)

    x = Flatten()(x)

    x = Dense(256, activation='relu')(x)
    if drop:
        x = Dropout(0.5)(x)

    outputs = Dense(100, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model


# =========================
# 🔥 TRAIN FUNCTION
# =========================
def train_model(model, name):
    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\nTraining {name}...")

    history = model.fit(
        x_train_main, y_train_main,
        epochs=20,
        batch_size=128,
        validation_data=(x_val, y_val),
        verbose=1
    )

    loss, acc = model.evaluate(x_test, y_test)
    print(f"{name} Test Accuracy:", acc)

    return acc


# =========================
# 🚀 RUN EXPERIMENT
# =========================

results = {}

# LeNet
results['LeNet_no_dropout'] = train_model(build_lenet(False), "LeNet No Dropout")
results['LeNet_dropout'] = train_model(build_lenet(True), "LeNet Dropout")

# ZFNet
results['ZFNet_no_dropout'] = train_model(build_zfnet(False), "ZFNet No Dropout")
results['ZFNet_dropout'] = train_model(build_zfnet(True), "ZFNet Dropout")

# ResNet
results['ResNet_no_dropout'] = train_model(build_resnet(False), "ResNet No Dropout")
results['ResNet_dropout'] = train_model(build_resnet(True), "ResNet Dropout")


print("\n📊 FINAL RESULTS:")
for k, v in results.items():
    print(k, ":", v)


169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training LeNet No Dropout...
Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 55s 170ms/step - accuracy: 0.0265 - loss: 4.5103 - val_accuracy: 0.0440 - val_loss: 4.3497
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 47s 150ms/step - accuracy: 0.0758 - loss: 4.1506 - val_accuracy: 0.0962 - val_loss: 4.0300
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 84s 156ms/step - accuracy: 0.1133 - loss: 3.9209 - val_accuracy: 0.1232 - val_loss: 3.8733
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 82s 157ms/step - accuracy: 0.1385 - loss: 3.7768 - val_accuracy: 0.1501 - val_loss: 3.7437
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 51s 162ms/step - accuracy: 0.1568 - loss: 3.6700 - val_accuracy: 0.1573 - val_loss: 3.6628
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 55s 177ms/step - accuracy: 0.1715 - loss: 3.5790 - val_accuracy: 0.1660 - val_loss: 3.5957
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 76s 157ms/step - accuracy: 0.1850 - loss: 3.5018 - val_accuracy: 0.1796 - val_loss: 3.5320
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 81s 154ms/ste

KeyboardInterrupt: 

“Dropout does not always depend on how deep the model is, but on how the model is designed. In our results, dropout reduced accuracy in LeNet and ResNet, but improved accuracy in ZFNet. This means dropout works best for medium-sized models that do not already have strong techniques to control overfitting. In models like ResNet, which already use batch normalization and skip connections, adding dropout can disturb learning and reduce performance.”




In [ ]:
import keras
from keras import datasets
from keras.models import Sequential, Model
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input, Add, Activation
from keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load dataset
(x_train, y_train), (x_test, y_test) = datasets.cifar100.load_data()

# Split
x_train_main = x_train[:40000] / 255.0
y_train_main = to_categorical(y_train[:40000], 100)

x_val = x_train[40000:] / 255.0
y_val = to_categorical(y_train[40000:], 100)

x_test = x_test / 255.0
y_test = to_categorical(y_test, 100)

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
datagen.fit(x_train_main)

# 🔹 LeNet Model
def create_lenet():
    model = Sequential()
    model.add(Conv2D(6, (5,5), activation='relu', input_shape=(32,32,3)))
    model.add(MaxPooling2D((2,2)))
    model.add(Conv2D(16, (5,5), activation='relu'))
    model.add(MaxPooling2D((2,2)))
    model.add(Flatten())
    model.add(Dense(120, activation='relu'))
    model.add(Dense(84, activation='relu'))
    model.add(Dense(100, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# 🔹 Simple ResNet Block
def res_block(x, filters):
    shortcut = x
    x = Conv2D(filters, (3,3), padding='same', activation='relu')(x)
    x = Conv2D(filters, (3,3), padding='same')(x)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

# 🔹 ResNet Model
def create_resnet():
    inputs = Input(shape=(32,32,3))
    x = Conv2D(32, (3,3), padding='same', activation='relu')(inputs)

    x = res_block(x, 32)
    x = res_block(x, 32)

    x = MaxPooling2D((2,2))(x)

    x = Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = res_block(x, 64)
    x = res_block(x, 64)

    x = MaxPooling2D((2,2))(x)

    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    outputs = Dense(100, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# 🔹 TRAINING FUNCTION
def train_model(model, use_aug):
    if use_aug:
        model.fit(datagen.flow(x_train_main, y_train_main, batch_size=128),
                  epochs=20,
                  validation_data=(x_val, y_val),
                  verbose=1)
    else:
        model.fit(x_train_main, y_train_main,
                  epochs=20,
                  batch_size=128,
                  validation_data=(x_val, y_val),
                  verbose=1)

    return model.evaluate(x_test, y_test, verbose=0)[1]

# 🔥 Run Experiments
lenet_no_aug = train_model(create_lenet(), False)
lenet_aug = train_model(create_lenet(), True)

resnet_no_aug = train_model(create_resnet(), False)
resnet_aug = train_model(create_resnet(), True)

# Results
print("LeNet without augmentation:", lenet_no_aug)
print("LeNet with augmentation:", lenet_aug)

print("ResNet without augmentation:", resnet_no_aug)
print("ResNet with augmentation:", resnet_aug)


“The effectiveness of data augmentation depends on the model’s ability to utilize the augmented data. In this experiment, augmentation reduced the performance of LeNet but significantly improved ResNet. This suggests that deeper models benefit more from augmented data because they have the capacity to learn complex patterns, whereas shallow models may fail to effectively learn from increased data variability.”
